[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C48_Cloud_Deployment_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯标准库 / numpy、CPU 可跑**，用 **Python 模拟控制平面** 复现云原生部署的机制，再用 **SLO 账 + 容量账 + 成本账** 把生产问题量化。

这个 notebook 做四件事：① 确认环境；② 用一个最小例子体会 **声明式控制循环（reconcile）**——K8s 的心脏；③ 立下三条纪律：**对拍 / 算账 / 下界思维**；④ 用错误预算把「99.9% 可用」翻译成人能理解的分钟数。

## 1 · 环境自检

只需要 `numpy`；`math`/`random`/`dataclasses`/`collections` 都是标准库。`pandas` 可选。

In [ ]:
import sys, platform, math, random, collections
from dataclasses import dataclass, field
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np; print('numpy', np.__version__)
try:
    import pandas as pd; print('pandas', pd.__version__, '(可选)')
except Exception:
    print('pandas 未安装（可选，不影响课程）')
print('环境就绪 ✅  —— 本课不需要 Docker / Kubernetes / GPU / 联网')

## 2 · 声明式控制循环：Kubernetes 的心脏

K8s 里几乎每个组件都是同一个模式：**看一眼期望状态（spec）、看一眼当前状态（status）、发出让二者靠拢的动作**。这个循环叫 **reconcile（调谐）**。

它和「命令式」的区别是本质性的：命令式说「启动 3 个进程」（说完就完了，挂了没人管）；声明式说「我要 3 个副本」（控制器**持续**保证这件事成立）。

下面用 30 行写一个最小的 Deployment 控制器，然后验证它的两个关键性质：**收敛** 与 **自愈**。

In [ ]:
def reconcile(current, desired):
    '''最小控制器：返回本轮要执行的动作列表。
       current: 当前存活的副本 id 集合；desired: 期望副本数。'''
    actions = []
    n = len(current)
    if n < desired:
        for _ in range(desired - n):
            actions.append(('create', None))
    elif n > desired:
        # 缩容时删除 id 最大的（真实 K8s 有更复杂的删除代价排序）
        for pod in sorted(current)[desired:]:
            actions.append(('delete', pod))
    return actions

def apply(current, actions, next_id):
    for kind, pod in actions:
        if kind == 'create':
            current.add(next_id); next_id += 1
        else:
            current.discard(pod)
    return current, next_id

# 从 0 个副本开始，期望 3 个
current, next_id, DESIRED = set(), 0, 3
history = []
for tick in range(5):
    acts = reconcile(current, DESIRED)
    current, next_id = apply(current, acts, next_id)
    history.append(len(current))
print('副本数演化:', history)
assert history[-1] == DESIRED, '控制器必须收敛到期望副本数'
assert history == sorted(history), '从 0 起步应单调增长到目标，不该过冲'
print('✅ 收敛性：控制器把 0 个副本调谐到', DESIRED, '个')

**自愈**才是声明式的真正威力：外部事件（节点宕机）把副本干掉后，控制器**不需要任何人下命令**就会补上。

下面模拟「随机杀 Pod」的混沌实验，验证系统始终回到期望状态。

In [ ]:
rng = random.Random(0)
current, next_id = {0, 1, 2}, 3
recovered_ticks = []
for tick in range(30):
    # 外部扰动：10% 概率随机杀掉一个副本（模拟节点宕机 / OOMKilled）
    if current and rng.random() < 0.3:
        victim = rng.choice(sorted(current)); current.discard(victim)
    # 控制器调谐
    current, next_id = apply(current, reconcile(current, DESIRED), next_id)
    recovered_ticks.append(len(current))

assert all(n == DESIRED for n in recovered_ticks), '每轮结束时都应已恢复到期望副本数'
print(f'30 轮混沌注入后，每轮末副本数恒为 {DESIRED} ✅')
print('这就是自愈：没有人下过一条「重启」命令，是控制循环在持续拉平差距。')

> **心智模型（贯穿全课）**：K8s 不是「一个能跑容器的系统」，而是**一组持续把现实拽向期望的控制循环**。
> Deployment 控制器管副本数、Service 控制器管端点列表、HPA 管副本数的期望值本身、调度器管 Pod 落到哪个节点。
> 你在模块 03/04 会把这些循环一个个写出来。

## 3 · SLO 账：把「99.9% 可用」翻译成分钟

生产的第一笔账。可用性目标（SLO）听起来抽象，但它等价于一个非常具体的数字：**这个月你允许坏多少分钟**。这叫 **错误预算（error budget）**。

In [ ]:
def error_budget_minutes(slo, window_days=30):
    '''给定可用性 SLO（如 0.999），返回窗口内允许的不可用分钟数。'''
    return (1 - slo) * window_days * 24 * 60

for slo in [0.99, 0.995, 0.999, 0.9999]:
    m = error_budget_minutes(slo)
    print(f'SLO {slo*100:>7.2f}%  -> 每 30 天允许不可用 {m:>8.1f} 分钟 ({m/60:.2f} 小时)')

assert abs(error_budget_minutes(0.999) - 43.2) < 0.1
# 关键直觉：每多一个 9，预算缩小 10 倍
assert error_budget_minutes(0.99) / error_budget_minutes(0.999) == 10
print('\n✅ 每多一个 9，错误预算缩小 10 倍——这就是为什么 SLO 是成本决策而不是技术决策。')

**错误预算的用法**（Google SRE 的核心实践）：它是**发布速度与稳定性之间的仲裁者**。

- 预算还剩很多 → 可以激进发布、多做实验；
- 预算烧完了 → 冻结发布，先修稳定性。

下面把「一次糟糕的发布」翻译成预算消耗，你会发现 **43 分钟其实非常短**。

In [ ]:
BUDGET = error_budget_minutes(0.999)      # 43.2 分钟/月

incidents = [
    ('一次全量发布引入 bug，5 分钟发现 + 10 分钟回滚', 15, 1.0),   # (描述, 分钟, 受影响流量比例)
    ('金丝雀发布同样的 bug，5% 流量，30 分钟才发现',    30, 0.05),
    ('一个节点宕机，8 副本少 1 个，无用户可见错误',      20, 0.0),
]
print(f'月度错误预算: {BUDGET:.1f} 分钟\n')
spent = 0.0
for desc, minutes, frac in incidents:
    cost = minutes * frac                  # 按受影响流量折算
    spent += cost
    print(f'  {desc}\n    -> 消耗 {cost:.2f} 分钟预算 ({cost/BUDGET*100:.1f}%)')
print(f'\n合计消耗 {spent:.2f} / {BUDGET:.1f} 分钟 = {spent/BUDGET*100:.1f}%')

# 核心对比：同一个 bug，全量发布 vs 金丝雀
assert 15 * 1.0 > 30 * 0.05, '全量发布 15 分钟比金丝雀 30 分钟更伤——爆炸半径压倒 MTTR'
print('\n✅ 关键结论：同一个 bug，全量 15 分钟(15.0) 比金丝雀 30 分钟(1.5) 贵 10 倍。')
print('   **爆炸半径 × 时长 = 损失**。模块 04 整章都在压这个乘积的第一项。')

## 4 · 容量账：一个副本能扛多少 QPS？

第二笔账。直觉上「服务能处理 100 QPS 就配 100 QPS 的流量」是**灾难性错误**——排队论告诉你，当利用率 ρ 逼近 1 时，等待时间会**爆炸式**增长。

先用最简单的 M/M/1 感受这条曲线（模块 02 会做完整的 M/M/c）。

In [ ]:
def mm1_wait(lam, mu):
    '''M/M/1 平均排队等待时间（不含服务时间）。lam 到达率, mu 服务率。'''
    rho = lam / mu
    if rho >= 1:
        return float('inf')
    return rho / (mu - lam)

MU = 10.0        # 单副本每秒能处理 10 个请求
print(f"{'利用率 ρ':>10s} {'到达率 λ':>10s} {'平均等待(ms)':>14s}")
for rho in [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]:
    lam = rho * MU
    print(f'{rho:>10.2f} {lam:>10.1f} {mm1_wait(lam, MU)*1000:>14.1f}')

w80, w95 = mm1_wait(0.80*MU, MU), mm1_wait(0.95*MU, MU)
assert w95 > 3 * w80, 'ρ 从 0.80 到 0.95，等待时间应急剧恶化'
print(f'\n✅ ρ: 0.80 -> 0.95（多榨 19% 吞吐），等待时间涨 {w95/w80:.1f} 倍。')
print('   这就是为什么生产容量规划的目标利用率通常是 0.6~0.8，而不是 0.95。')

## 5 · 成本账：$ / 1M token

第三笔账，也是最容易被工程师忽略、却最容易被老板问到的一笔。把实例小时价翻译成**单位经济（unit economics）**。

In [ ]:
def cost_per_million_tokens(hourly_usd, tokens_per_sec, utilization=1.0):
    '''把实例小时价换算成每百万输出 token 的成本。'''
    tokens_per_hour = tokens_per_sec * 3600 * utilization
    if tokens_per_hour == 0:
        return float('inf')
    return hourly_usd / (tokens_per_hour / 1e6)

# 公开量级：8×H100 节点按需约 $30~40/h；7B 模型在其上可达数千 tok/s 总吞吐
H100_NODE_HOURLY = 32.0
for tps, util, label in [(4000, 1.00, '满载'),
                         (4000, 0.60, '平均利用率 60%'),
                         (4000, 0.25, '夜间低谷 25%')]:
    c = cost_per_million_tokens(H100_NODE_HOURLY, tps, util)
    print(f'{label:>18s}: ${c:>7.3f} / 1M tokens')

c_full = cost_per_million_tokens(H100_NODE_HOURLY, 4000, 1.0)
c_qtr  = cost_per_million_tokens(H100_NODE_HOURLY, 4000, 0.25)
assert abs(c_qtr / c_full - 4.0) < 1e-9, '利用率降到 1/4，单位成本应涨 4 倍'
print('\n✅ 单位成本与利用率成反比。**闲置的 GPU 是全额计费的**——')
print('   这一条就解释了模块 04 的自动扩缩和模块 05 的 spot 为什么值得做。')

## 6 · 立纪律三：下界思维（blast radius）

前两条纪律（对拍、算账）你在 C43/C39 已经见过。本课加第三条，它是生产工程独有的：

> **对每个设计问一句：最坏情况下，多少用户受影响、多久恢复？**

把它封装成一个小工具，后面每个模块都会用它评估方案。

In [ ]:
@dataclass
class Blast:
    name: str
    affected_frac: float     # 受影响流量比例
    detect_min: float        # 平均发现时间
    recover_min: float       # 发现后恢复时间

    @property
    def budget_cost(self):   # 消耗的错误预算分钟数
        return self.affected_frac * (self.detect_min + self.recover_min)

strategies = [
    Blast('全量发布 (recreate)',      1.00, 5, 10),
    Blast('滚动更新 (25% surge)',     0.25, 5, 10),
    Blast('金丝雀 5% + 自动分析',      0.05, 3,  2),
    Blast('影子流量 (不影响用户)',      0.00, 60, 0),
]
print(f"{'策略':<26s} {'受影响':>8s} {'MTTR(min)':>10s} {'预算消耗':>10s}")
for s in strategies:
    print(f'{s.name:<26s} {s.affected_frac:>8.0%} {s.detect_min+s.recover_min:>10.0f} {s.budget_cost:>10.2f}')

costs = [s.budget_cost for s in strategies]
assert costs == sorted(costs, reverse=True), '爆炸半径越小，预算消耗应越低'
assert strategies[-1].budget_cost == 0, '影子流量对用户零影响，即使跑一小时'
print('\n✅ 影子流量发现慢 20 倍，但预算消耗为 0——**慢而安全 > 快而全量**。')

## 7 · ✏️ 练习：把三笔账合成一个决策

你现在是这个服务的负责人。给定：SLO=99.9%、单副本 μ=10 QPS、节点 $32/h。

实现 `plan_capacity(peak_qps, mu, target_rho)`：返回 **(副本数, 实际利用率)**。
要求：副本数是**满足目标利用率的最小整数**，即 `n = ceil(peak_qps / (mu * target_rho))`，实际利用率 `= peak_qps / (n * mu)`。

In [ ]:
def plan_capacity(peak_qps, mu, target_rho):
    # TODO: 返回 (n_replicas, actual_rho)
    #   n = ceil(peak_qps / (mu * target_rho))，actual_rho = peak_qps / (n * mu)
    raise NotImplementedError

In [ ]:
# —— 练习自测 ——
n, rho = plan_capacity(100, 10, 0.8)
assert n == 13, f'100 QPS / (10*0.8) = 12.5 -> 向上取整 13，得到 {n}'
assert abs(rho - 100/130) < 1e-9, '实际利用率应为 100/(13*10)'
assert rho <= 0.8 + 1e-9, '实际利用率不应超过目标'

# 目标利用率越低（越保守），需要的副本越多
n_safe, _ = plan_capacity(100, 10, 0.6)
n_greedy, _ = plan_capacity(100, 10, 0.95)
assert n_safe > n_greedy, '更保守的目标利用率需要更多副本'
print(f'peak=100 QPS: ρ*=0.6 需 {n_safe} 副本 | ρ*=0.8 需 {n} 副本 | ρ*=0.95 需 {n_greedy} 副本')
print('✅ 练习通过：容量规划 = 在「延迟风险」与「成本」之间选一个 ρ*')

---
### 📖 参考答案

In [ ]:
def plan_capacity(peak_qps, mu, target_rho):
    n = math.ceil(peak_qps / (mu * target_rho))
    return n, peak_qps / (n * mu)

## 8 · 🧪 胶囊：三笔账串成一张决策表

把 SLO / 容量 / 成本三笔账放在一起，你会看到它们如何**互相拉扯**——这正是生产决策的真实形态。

In [ ]:
PEAK_QPS, MU, NODE_HOURLY, REPLICAS_PER_NODE = 100, 10.0, 32.0, 4

print(f"{'目标ρ':>7s} {'副本':>5s} {'节点':>5s} {'月成本$':>10s} {'M/M/1等待ms':>13s}")
rows = []
for target in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    n = math.ceil(PEAK_QPS / (MU * target))
    nodes = math.ceil(n / REPLICAS_PER_NODE)
    monthly = nodes * NODE_HOURLY * 24 * 30
    rho = PEAK_QPS / (n * MU)
    # 每副本视作独立 M/M/1，到达率均分
    wait_ms = mm1_wait(PEAK_QPS / n, MU) * 1000
    rows.append((target, n, nodes, monthly, wait_ms))
    print(f'{target:>7.2f} {n:>5d} {nodes:>5d} {monthly:>10,.0f} {wait_ms:>13.1f}')

# 成本随目标利用率单调不增；等待时间随目标利用率单调不减
costs = [r[3] for r in rows]; waits = [r[4] for r in rows]
assert costs == sorted(costs, reverse=True), '目标利用率越高，越省钱'
assert waits == sorted(waits), '目标利用率越高，等待越长'
print('\n✅ 三笔账的张力：省钱(高ρ) 与 低延迟(低ρ) 直接对立。')
print('   没有「最优」，只有「给定 SLO 下的最省」——这就是容量规划的全部内容。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你写的每个部署机制（镜像层缓存、请求路由、装箱调度、滚动更新、HPA、成本模型）都会
① 与朴素参考**对拍**确认正确，② 用 SLO/容量/成本**算账**量化，③ 用**爆炸半径**评估最坏情况。

**接下来五个模块**：01 容器化 → 02 推理服务 API → 03 Kubernetes 编排 → 04 发布与自动扩缩 → 05 云平台·调度·成本。

下一站：**模块 01 · 容器化** —— 先把「在我机器上能跑」这句话彻底消灭掉。